# Week 10 — Support Vector Machines & Kernel Methods
## Notebook 6: The Kernel Trick — Finding Separability in Higher Dimensions

| | |
|---|---|
| **Difficulty** | ⭐⭐⭐⭐⭐ (Advanced) |
| **Estimated Time** | 3 hours |
| **Theme** | Email Spam Classification |
| **Prerequisites** | Notebook 05 (Support Vectors) |

## Why This Matters

Real-world spam isn't linearly separable. A spammer who knows your filter exploits non-linear patterns: disguising keywords with numbers ("fr3e m0ney"), combining safe words in suspicious sequences, or embedding spam in image attachments. A linear decision boundary — no matter how carefully positioned — cannot capture these curvilinear patterns.

The kernel trick is arguably the most elegant idea in classical machine learning. It lets you:
1. Work in an (effectively) infinite-dimensional feature space
2. Pay only the computational cost of evaluating a simple similarity function
3. Find non-linear decision boundaries without ever explicitly computing the transformed features

This is not a computational shortcut. It is a **mathematical equivalence** — and understanding it deeply will make you a better ML practitioner.

## Real-World Analogy: The Marble Bowl

Imagine red and blue marbles mixed together in a salad bowl — the reds clustered in the center, the blues arranged around them. Try to separate them with a flat ruler (a 2D line): impossible. They are not linearly separable.

Now flip the bowl upside down over a table. Marbles fly up into the air. In 3D space, they separate: the reds (originally in the center) go higher because they had more upward momentum; the blues (on the rim) scatter lower. Now you CAN slide a flat sheet of paper between them — a plane that separates the classes in 3D.

**The kernel trick:** You don't physically move the marbles into 3D. Instead, you change the *similarity metric* you use to compare marbles. The Gaussian (RBF) kernel measures similarity in a way that implicitly corresponds to the infinite-dimensional "flipped" space — without ever computing the coordinates in that space.

For spam:
- In the original 2D feature space: spam and legitimate emails overlap (not separable with a line)
- After the implicit mapping via a kernel: they separate perfectly — you can find a hyperplane (in the higher-dimensional space) that classifies them with zero error
- The kernel computes this mapping implicitly: you never compute the 1000-dimensional feature vector

## The Problem: Linear SVM Fails on Non-Linear Data

Consider an email classification problem where spam and legitimate emails have an *annular* (ring-shaped) structure:
- Inner ring: spam emails with moderate keyword counts across all categories
- Outer ring: legitimate emails that are topic-specific (very high or very low on specific dimensions)

No straight line can separate these. The decision boundary must be **curved**.

### Solution 1 (Naive): Engineer New Features

Add quadratic features manually: for inputs $(x_1, x_2)$, create $(x_1, x_2, x_1^2, x_2^2, x_1 x_2)$. Now fit a linear SVM in this 5D space. A linear boundary in 5D can be a curved surface in 2D!

**Problem:** With $d=10,000$ email features (words), a polynomial of degree 2 gives $\binom{10002}{2} \approx 50$ million features. Degree 3: ~$10^{12}$ features. Completely infeasible.

### Solution 2 (Kernel Trick): Compute Dot Products Implicitly

Notice that the SVM dual objective uses data **only through dot products** $\mathbf{x}_i \cdot \mathbf{x}_j$. The key identity:

$$K(\mathbf{x}, \mathbf{z}) = \phi(\mathbf{x}) \cdot \phi(\mathbf{z})$$

If we can compute $K(\mathbf{x}, \mathbf{z})$ directly (without computing $\phi(\mathbf{x})$ and $\phi(\mathbf{z})$ explicitly), we can run SVM in the high-dimensional space at the cost of evaluating $K$ — just one simple formula.

## The Math

### Standard Kernels

| Kernel | Formula | Feature Space |
|---|---|---|
| Linear | $K(\mathbf{x}, \mathbf{z}) = \mathbf{x}\cdot\mathbf{z}$ | Original space (identity) |
| Polynomial | $K(\mathbf{x}, \mathbf{z}) = (\mathbf{x}\cdot\mathbf{z} + c)^d$ | All degree-$d$ polynomial features |
| RBF (Gaussian) | $K(\mathbf{x}, \mathbf{z}) = \exp(-\gamma\|\mathbf{x}-\mathbf{z}\|^2)$ | **Infinite**-dimensional |
| Sigmoid | $K(\mathbf{x}, \mathbf{z}) = \tanh(\alpha\,\mathbf{x}\cdot\mathbf{z} + c)$ | Conditionally PSD |

### Why K(x,z) = (x·z + 1)² is valid — expand it:

For $\mathbf{x} = (x_1, x_2)$ and $\mathbf{z} = (z_1, z_2)$:

$$(\mathbf{x}\cdot\mathbf{z} + 1)^2 = (x_1 z_1 + x_2 z_2 + 1)^2 = x_1^2 z_1^2 + x_2^2 z_2^2 + 1 + 2x_1 x_2 z_1 z_2 + 2x_1 z_1 + 2x_2 z_2$$

This equals the dot product of:
$$\phi(\mathbf{x}) = (x_1^2,\ x_2^2,\ 1,\ \sqrt{2}x_1 x_2,\ \sqrt{2}x_1,\ \sqrt{2}x_2)$$

A **6-dimensional** feature map, computed with a single $(\cdot + 1)^2$ evaluation!

### Mercer's Theorem (Validity Condition)

A function $K(\mathbf{x}, \mathbf{z})$ is a valid kernel **if and only if** the Gram matrix:
$$G_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$$
is **positive semi-definite** (PSD) for all possible datasets $\{\mathbf{x}_1, \ldots, \mathbf{x}_n\}$.

PSD means: $\mathbf{c}^\top G\, \mathbf{c} \geq 0$ for all vectors $\mathbf{c} \in \mathbb{R}^n$.

### Kernelized Dual

Replace every $\mathbf{x}_i \cdot \mathbf{x}_j$ with $K(\mathbf{x}_i, \mathbf{x}_j)$:

$$\max_{\boldsymbol{\alpha}} \sum_i \alpha_i - \frac{1}{2}\sum_{i,j}\alpha_i\alpha_j y_i y_j\, K(\mathbf{x}_i, \mathbf{x}_j)$$

Prediction becomes:
$$f(\mathbf{x}) = \text{sign}\left(\sum_{i \in \text{SVs}} \alpha_i y_i\, K(\mathbf{x}_i, \mathbf{x}) + b\right)$$

## Setup & Non-Linear Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from sklearn.svm import SVC
from sklearn.datasets import make_circles
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Non-linearly separable spam dataset: concentric circles ──────────────
# Inner ring  (+1): borderline spam (moderate keyword density, moderate link ratio)
# Outer ring  (-1): clearly legitimate (very high or very low on features)
#
# make_circles: inner circle = class 1 (spam), outer = class 0 → remap to -1

X_raw, y_raw = make_circles(n_samples=300, noise=0.08, factor=0.45, random_state=42)
y_circles    = np.where(y_raw == 1, 1, -1)  # inner=spam(+1), outer=legit(-1)

# Scale to interpretable feature ranges
scaler = StandardScaler()
X_circles = scaler.fit_transform(X_raw)

print(f"Dataset shape : {X_circles.shape}")
print(f"Classes       : Spam (inner, +1) = {(y_circles==1).sum()}  |  Legit (outer, -1) = {(y_circles==-1).sum()}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X_circles[y_circles==1,  0], X_circles[y_circles==1,  1],
           c='#DD8452', edgecolors='k', s=35, linewidths=0.6, label='Spam (inner)', zorder=3)
ax.scatter(X_circles[y_circles==-1, 0], X_circles[y_circles==-1, 1],
           c='#4C72B0', edgecolors='k', s=35, linewidths=0.6, label='Legitimate (outer)', zorder=3)
ax.set_title('Email Spam Dataset\n(Non-Linearly Separable — Concentric Structure)', fontsize=12)
ax.set_xlabel('Feature 1 (Keyword Pattern Score)')
ax.set_ylabel('Feature 2 (Link Behaviour Score)')
ax.legend()
plt.tight_layout()
plt.show()

## Section 1: Linear SVM Fails — Demonstrating the Problem

In [ ]:
# ── Train linear SVM and show it fails ──────────────────────────────────
clf_linear = SVC(kernel='linear', C=1.0, random_state=42)
clf_linear.fit(X_circles, y_circles)
acc_linear = accuracy_score(y_circles, clf_linear.predict(X_circles))

print(f"Linear SVM Training Accuracy: {acc_linear:.1%}")
print(f"  (Chance level = 50%)")
print(f"  Number of support vectors: {clf_linear.n_support_.sum()}")

# Visualize the failing linear boundary
def plot_decision_boundary_2d(ax, model, X, y, title, cmap_bg=True):
    h = 0.03
    x_min, x_max = X[:,0].min()-0.4, X[:,0].max()+0.4
    y_min, y_max = X[:,1].min()-0.4, X[:,1].max()+0.4
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    if cmap_bg:
        ax.contourf(xx, yy, Z, alpha=0.15, levels=[-2,-0.5,0.5,2],
                    colors=['#4C72B0','white','#DD8452'])
    ax.contour(xx, yy, Z, levels=[0], colors='black', linewidths=2)
    ax.scatter(X[y==1, 0],  X[y==1, 1],  c='#DD8452', edgecolors='k',
               s=35, linewidths=0.6, label='Spam',       alpha=0.8, zorder=3)
    ax.scatter(X[y==-1,0], X[y==-1,1], c='#4C72B0', edgecolors='k',
               s=35, linewidths=0.6, label='Legitimate', alpha=0.8, zorder=3)
    sv = model.support_vectors_
    ax.scatter(sv[:,0], sv[:,1], s=130, facecolors='none',
               edgecolors='red', linewidths=1.8, zorder=4, label='SVs')
    acc = accuracy_score(y, model.predict(X))
    ax.set_title(f"{title}\nAccuracy: {acc:.1%}", fontsize=11)
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
    ax.legend(fontsize=8)

fig, ax = plt.subplots(figsize=(6.5, 5))
plot_decision_boundary_2d(ax, clf_linear, X_circles, y_circles,
                          'Linear SVM — Cannot Separate Concentric Data')
plt.tight_layout()
plt.show()

print("\nA straight line cannot separate concentric rings.")
print("We need a curved decision boundary — enter the kernel trick.")

## Section 2: Explicit Feature Mapping — Lifting to 3D

In [ ]:
# ── Explicit phi: R² → R³ ─────────────────────────────────────────────────
# phi(x1, x2) = (x1², sqrt(2)*x1*x2, x2²)
# This corresponds to the degree-2 polynomial kernel with c=0

def phi_quadratic(X):
    """Map R² → R³ via (x1, x2) → (x1², √2·x1·x2, x2²)"""
    x1, x2 = X[:, 0], X[:, 1]
    return np.column_stack([
        x1**2,
        np.sqrt(2) * x1 * x2,
        x2**2
    ])

X_3d = phi_quadratic(X_circles)

print("Explicit mapping: R² → R³")
print(f"  φ(x₁, x₂) = (x₁², √2·x₁·x₂, x₂²)")
print(f"  Original shape : {X_circles.shape}")
print(f"  Mapped shape   : {X_3d.shape}")
print(f"\nFirst 3 points (original → mapped):")
for i in range(3):
    print(f"  ({X_circles[i,0]:.3f}, {X_circles[i,1]:.3f})  →  "
          f"({X_3d[i,0]:.4f}, {X_3d[i,1]:.4f}, {X_3d[i,2]:.4f})")

# ── 3D visualization ──────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 5))

# Left: 2D original
ax2d = fig.add_subplot(121)
ax2d.scatter(X_circles[y_circles==1, 0],  X_circles[y_circles==1, 1],
             c='#DD8452', edgecolors='k', s=35, linewidths=0.6, label='Spam', alpha=0.8)
ax2d.scatter(X_circles[y_circles==-1, 0], X_circles[y_circles==-1, 1],
             c='#4C72B0', edgecolors='k', s=35, linewidths=0.6, label='Legitimate', alpha=0.8)
ax2d.set_title('Original 2D Feature Space\n(Not Linearly Separable)', fontsize=11)
ax2d.set_xlabel('x₁'); ax2d.set_ylabel('x₂')
ax2d.legend()

# Right: 3D mapped
ax3d = fig.add_subplot(122, projection='3d')
ax3d.scatter(X_3d[y_circles==1, 0],  X_3d[y_circles==1, 1],  X_3d[y_circles==1, 2],
             c='#DD8452', edgecolors='k', s=40, linewidths=0.4, alpha=0.8, label='Spam')
ax3d.scatter(X_3d[y_circles==-1, 0], X_3d[y_circles==-1, 1], X_3d[y_circles==-1, 2],
             c='#4C72B0', edgecolors='k', s=40, linewidths=0.4, alpha=0.8, label='Legitimate')

# Draw a separating plane (fitted from the 3D data)
from sklearn.svm import SVC as SVC3
clf_3d = SVC3(kernel='linear', C=1.0).fit(X_3d, y_circles)
w = clf_3d.coef_[0]; b_3d = clf_3d.intercept_[0]
# w[0]*z1 + w[1]*z2 + w[2]*z3 + b = 0  →  z3 = -(w[0]*z1 + w[1]*z2 + b) / w[2]
z1_g = np.linspace(X_3d[:,0].min(), X_3d[:,0].max(), 25)
z2_g = np.linspace(X_3d[:,1].min(), X_3d[:,1].max(), 25)
ZZ1, ZZ2 = np.meshgrid(z1_g, z2_g)
ZZ3 = -(w[0]*ZZ1 + w[1]*ZZ2 + b_3d) / w[2]
ax3d.plot_surface(ZZ1, ZZ2, ZZ3, alpha=0.25, color='lime')
acc_3d = accuracy_score(y_circles, clf_3d.predict(X_3d))
ax3d.set_title(f'Mapped 3D Space φ(x₁,x₂) = (x₁², √2·x₁x₂, x₂²)\n'
               f'Linearly Separable! (Accuracy: {acc_3d:.1%})', fontsize=10)
ax3d.set_xlabel('x₁²'); ax3d.set_ylabel('√2·x₁x₂'); ax3d.set_zlabel('x₂²')
ax3d.legend(fontsize=8)

fig.suptitle('Explicit Feature Mapping: 2D → 3D Creates Linear Separability',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 3: The Kernel Trick — Same Result, No Explicit Mapping

We verify the key identity:
$$K(\mathbf{x}, \mathbf{z}) = (\mathbf{x}\cdot\mathbf{z})^2 = \phi(\mathbf{x})\cdot\phi(\mathbf{z})$$

where $\phi(\mathbf{x}) = (x_1^2, \sqrt{2}x_1 x_2, x_2^2)$.

In [ ]:
# ── Verify: K(x,z) = (x·z)² = φ(x)·φ(z) ─────────────────────────────────
np.random.seed(42)
x_test = X_circles[:5]
z_test = X_circles[5:10]

print("Verification: K(x,z) = (x·z)² vs φ(x)·φ(z)")
print(f"{'i':>3} {'j':>3} | {'(x·z)²':>12} | {'φ(x)·φ(z)':>12} | {'|Δ|':>10}")
print('-' * 52)
for i in range(5):
    for j in range(5):
        k_implicit = (x_test[i] @ z_test[j])**2       # kernel formula
        k_explicit = phi_quadratic(x_test[[i]]) @ phi_quadratic(z_test[[j]]).T  # dot in 3D
        print(f"{i:>3} {j:>3} | {k_implicit:>12.6f} | {k_explicit[0,0]:>12.6f} | "
              f"{abs(k_implicit - k_explicit[0,0]):>10.2e}")

print("\n→ They match exactly! The kernel computes the 3D dot product without constructing φ.")

# Now show: polynomial kernel (x·z + 1)² with sklearn
# This corresponds to a richer 6D feature space
print("\n─" * 40)
print("\nExpanding K(x,z) = (x·z + 1)² for 2D inputs:")
print("= x₁²z₁² + x₂²z₂² + 1 + 2x₁x₂z₁z₂ + 2x₁z₁ + 2x₂z₂")
print("= φ(x)·φ(z)  where  φ(x) = (x₁², x₂², 1, √2·x₁x₂, √2·x₁, √2·x₂)")
print("This is a 6-dimensional feature space — computed by a single squaring operation!")

## Section 4: sklearn SVC with Polynomial and RBF Kernels

In [ ]:
# ── Train kernel SVMs and compare decision boundaries ─────────────────────
kernels = [
    ('linear',    SVC(kernel='linear',                 C=1.0, random_state=42)),
    ('poly d=2',  SVC(kernel='poly', degree=2, coef0=1, C=1.0, random_state=42)),
    ('poly d=3',  SVC(kernel='poly', degree=3, coef0=1, C=1.0, random_state=42)),
    ('rbf γ=0.5', SVC(kernel='rbf', gamma=0.5,          C=1.0, random_state=42)),
    ('rbf γ=1.0', SVC(kernel='rbf', gamma=1.0,          C=1.0, random_state=42)),
    ('rbf γ=5.0', SVC(kernel='rbf', gamma=5.0,          C=1.0, random_state=42)),
]

for name, clf in kernels:
    clf.fit(X_circles, y_circles)

print(f"{'Kernel':>12} | {'Train Acc':>10} | {'# SVs':>6}")
print('-' * 36)
for name, clf in kernels:
    acc = accuracy_score(y_circles, clf.predict(X_circles))
    print(f"{name:>12} | {acc:>10.1%} | {clf.n_support_.sum():>6d}")

# ── 6-panel boundary visualization ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for ax, (name, clf) in zip(axes, kernels):
    plot_decision_boundary_2d(ax, clf, X_circles, y_circles, f'Kernel: {name}')

fig.suptitle('Kernel SVM Decision Boundaries on Concentric Spam Data',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Section 5: RBF Kernel — Gram Matrix Heatmap

The Gram matrix $G_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$ encodes pairwise similarity. For a good kernel, points of the same class should have high similarity (block structure), while cross-class similarity should be low.

In [ ]:
# ── Build Gram matrices for a subset of data ──────────────────────────────
# Take 50 points (25 per class), sorted by class for block structure
np.random.seed(42)
idx_spam  = np.where(y_circles ==  1)[0][:25]
idx_legit = np.where(y_circles == -1)[0][:25]
idx_sub   = np.concatenate([idx_spam, idx_legit])
X_sub     = X_circles[idx_sub]
y_sub     = y_circles[idx_sub]

def gram_matrix(X, kernel_fn):
    n = len(X)
    G = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            G[i, j] = kernel_fn(X[i], X[j])
    return G

def k_linear(x, z):  return x @ z
def k_poly2(x, z):   return (x @ z + 1)**2
def k_rbf(x, z, g=1.0): return np.exp(-g * np.sum((x-z)**2))

G_lin  = gram_matrix(X_sub, k_linear)
G_poly = gram_matrix(X_sub, k_poly2)
G_rbf  = gram_matrix(X_sub, lambda x,z: k_rbf(x,z,g=1.0))

# Check PSD (all eigenvalues ≥ 0)
for name, G in [('Linear', G_lin), ('Poly(d=2)', G_poly), ('RBF(γ=1)', G_rbf)]:
    eigvals = np.linalg.eigvalsh(G)
    print(f"{name:>12}: min eigenvalue = {eigvals.min():.4f}  →  "
          f"{'PSD ✓' if eigvals.min() >= -1e-8 else 'NOT PSD ✗'}")

# ── Heatmap visualization ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (title, G) in zip(axes, [
    ('Linear Kernel\nK(x,z) = x·z', G_lin),
    ('Polynomial Kernel d=2\nK(x,z) = (x·z + 1)²', G_poly),
    ('RBF Kernel γ=1\nK(x,z) = exp(-γ‖x-z‖²)', G_rbf)
]):
    sns.heatmap(G, ax=ax, cmap='RdYlGn', center=0,
                xticklabels=False, yticklabels=False,
                cbar_kws={'shrink': 0.8})
    ax.axhline(25, color='black', lw=2)
    ax.axvline(25, color='black', lw=2)
    ax.set_title(title, fontsize=11)
    ax.text(12.5, 52, 'Spam', ha='center', va='center', fontsize=10, color='white',
            fontweight='bold')
    ax.text(37.5, 52, 'Legit', ha='center', va='center', fontsize=10, color='white',
            fontweight='bold')
    ax.set_xlabel('Sample index (sorted by class)')
    ax.set_ylabel('Sample index (sorted by class)')

fig.suptitle('Gram Matrix Heatmaps: Block Structure = Good Class Separation\n'
             '(Top-left = spam×spam, bottom-right = legit×legit)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 6: Effect of γ on RBF Kernel — Overfitting vs Underfitting

In [ ]:
# ── RBF kernel: γ controls locality of similarity ─────────────────────────
# Small γ: two points similar even if far apart → smooth, broad boundary
# Large γ: only very close points similar → spiky, overfit boundary

gammas = [0.1, 0.5, 1.0, 3.0, 10.0, 50.0]

print(f"{'γ':>6} | {'Train Acc':>10} | {'# SVs':>7} | {'Boundary'}")
print('-' * 55)

rbf_models = {}
for g in gammas:
    clf = SVC(kernel='rbf', gamma=g, C=1.0, random_state=42)
    clf.fit(X_circles, y_circles)
    acc = accuracy_score(y_circles, clf.predict(X_circles))
    rbf_models[g] = clf
    char = 'Smooth' if g < 1 else ('Good' if g < 5 else 'Overfit')
    print(f"{g:>6.1f} | {acc:>10.1%} | {clf.n_support_.sum():>7d} | {char}")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for ax, g in zip(axes, gammas):
    clf = rbf_models[g]
    h = 0.03
    x_min, x_max = X_circles[:,0].min()-0.4, X_circles[:,0].max()+0.4
    y_min2, y_max2 = X_circles[:,1].min()-0.4, X_circles[:,1].max()+0.4
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min2, y_max2, h))
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.7)
    ax.contour(xx,  yy, Z, levels=[0], colors='black', linewidths=2)
    ax.scatter(X_circles[y_circles==1, 0],  X_circles[y_circles==1, 1],
               c='#DD8452', edgecolors='k', s=30, linewidths=0.5, alpha=0.9)
    ax.scatter(X_circles[y_circles==-1, 0], X_circles[y_circles==-1, 1],
               c='#4C72B0', edgecolors='k', s=30, linewidths=0.5, alpha=0.9)
    sv = clf.support_vectors_
    ax.scatter(sv[:,0], sv[:,1], s=120, facecolors='none',
               edgecolors='yellow', linewidths=1.8, zorder=4)
    acc = accuracy_score(y_circles, clf.predict(X_circles))
    ax.set_title(f'RBF  γ = {g}  |  Acc = {acc:.1%}  |  SVs = {clf.n_support_.sum()}',
                 fontsize=10)
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

fig.suptitle('RBF Kernel: Effect of γ on Decision Boundary\n'
             'Small γ = smooth (underfit)  |  Large γ = spiky (overfit)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 7: Verifying Mercer's Condition — PSD Gram Matrices

In [ ]:
# ── Check PSD condition for various kernels ───────────────────────────────
np.random.seed(42)
X_small = np.random.randn(30, 2)  # 30 random 2D points

def check_psd(G, name):
    eigvals = np.linalg.eigvalsh(G)
    is_psd  = eigvals.min() >= -1e-8
    print(f"  {name:<30} | min eigval = {eigvals.min():>9.4f} | "
          f"{'PSD (valid kernel)' if is_psd else 'NOT PSD (invalid kernel)'}")
    return eigvals

print("Mercer's Condition: Is the Gram Matrix Positive Semi-Definite?")
print("=" * 72)

# 1. Linear kernel
G_lin = X_small @ X_small.T
check_psd(G_lin, 'Linear  K(x,z)=x·z')

# 2. Polynomial degree 2
G_p2 = (X_small @ X_small.T + 1)**2
check_psd(G_p2, 'Poly d=2  (x·z+1)²')

# 3. Polynomial degree 3
G_p3 = (X_small @ X_small.T + 1)**3
check_psd(G_p3, 'Poly d=3  (x·z+1)³')

# 4. RBF
from sklearn.metrics.pairwise import rbf_kernel
G_rbf_check = rbf_kernel(X_small, gamma=1.0)
check_psd(G_rbf_check, 'RBF  exp(-‖x-z‖²)')

# 5. Exponential of dot product (is exp(x·z) valid?)
G_exp_dot = np.exp(X_small @ X_small.T)
check_psd(G_exp_dot, 'exp(x·z)  (Taylor series)')

# 6. Absolute difference — NOT a valid kernel
n = len(X_small)
G_abs = np.array([[np.abs(X_small[i] @ X_small[j]) for j in range(n)] for i in range(n)])
check_psd(G_abs, '|x·z|  (absolute value)')

# 7. Difference kernel — definitely not PSD
G_diff = np.array([[X_small[i] @ X_small[j] - 5.0 for j in range(n)] for i in range(n)])
check_psd(G_diff, 'x·z - 5  (shifted linear)')

print("\nKey: Sums, products, and exp() of valid kernels are also valid kernels.")
print("     The PSD condition ensures the QP dual always has a unique optimal solution.")

## Section 8: Comparing All Kernels — Comprehensive Decision Boundary Summary

In [ ]:
# ── Four-way comparison: linear vs poly(2) vs poly(3) vs rbf ─────────────
comparison_kernels = [
    ('Linear\n(C=1)',         SVC(kernel='linear',                  C=1.0, random_state=42)),
    ('Polynomial d=2\n(C=1)', SVC(kernel='poly', degree=2, coef0=1, C=1.0, random_state=42)),
    ('Polynomial d=3\n(C=1)', SVC(kernel='poly', degree=3, coef0=1, C=1.0, random_state=42)),
    ('RBF γ=1\n(C=1)',        SVC(kernel='rbf',  gamma=1.0,         C=1.0, random_state=42)),
]
for _, clf in comparison_kernels:
    clf.fit(X_circles, y_circles)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (name, clf) in zip(axes, comparison_kernels):
    plot_decision_boundary_2d(ax, clf, X_circles, y_circles, name)

fig.suptitle('Kernel Comparison on Concentric Spam Classification',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── RBF kernel similarity profile ─────────────────────────────────────────
print("\n--- RBF Similarity Profile ---")
x0 = np.array([0.0, 0.0])
distances = np.linspace(0, 4, 100)

fig, ax = plt.subplots(figsize=(8, 4))
for gamma in [0.2, 0.5, 1.0, 2.0, 5.0]:
    sims = np.exp(-gamma * distances**2)
    ax.plot(distances, sims, lw=2, label=f'γ = {gamma}')

ax.set_xlabel('Distance ‖x - z‖', fontsize=12)
ax.set_ylabel('RBF Similarity K(x,z)', fontsize=12)
ax.set_title('RBF Kernel Similarity vs Distance\n'
             'Large γ = similarity drops off quickly (local model)', fontsize=11)
ax.legend(fontsize=10)
ax.axhline(0.5, color='gray', ls='--', alpha=0.5, label='Similarity = 0.5')
plt.tight_layout()
plt.show()

## Section 9: Kernel SVM on a More Complex Spam Scenario

Let's test on a multi-cluster dataset that mimics real spam: legitimate emails cluster tightly around specific topics, while spam is more spread out and overlapping.

In [ ]:
# ── More complex multi-cluster dataset ───────────────────────────────────
from sklearn.datasets import make_moons
np.random.seed(42)

X_moon, y_moon_raw = make_moons(n_samples=250, noise=0.18, random_state=42)
y_moon = np.where(y_moon_raw == 1, 1, -1)

# Normalize
sc2 = StandardScaler()
X_moon = sc2.fit_transform(X_moon)

configs = [
    ('Linear',         SVC(kernel='linear',          C=1.0,  random_state=42)),
    ('RBF γ=0.3',     SVC(kernel='rbf', gamma=0.3,  C=1.0,  random_state=42)),
    ('RBF γ=1.0',     SVC(kernel='rbf', gamma=1.0,  C=1.0,  random_state=42)),
    ('RBF γ=5.0',     SVC(kernel='rbf', gamma=5.0,  C=1.0,  random_state=42)),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (name, clf) in zip(axes, configs):
    clf.fit(X_moon, y_moon)
    plot_decision_boundary_2d(ax, clf, X_moon, y_moon, f'{name}')

fig.suptitle('Kernel SVM on Moons Dataset (Non-Linear Spam Patterns)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
print(f"{'Kernel':>14} | {'Train Acc':>10} | {'# SVs':>7}")
print('-' * 38)
for name, clf in configs:
    acc = accuracy_score(y_moon, clf.predict(X_moon))
    print(f"{name:>14} | {acc:>10.1%} | {clf.n_support_.sum():>7d}")

## Self-Check Questions

These questions are harder than Notebook 5 — they require conceptual depth. Think carefully.

---

### Q1
The RBF kernel corresponds to an **infinite-dimensional** feature space. How can we train an SVM with an infinite-dimensional feature vector? Why is this computationally feasible?

<details>
<summary>Show Answer</summary>

**We never actually compute the infinite-dimensional feature vector $\phi(\mathbf{x})$.**

The SVM dual objective depends on the data only through **dot products** $\phi(\mathbf{x}_i)\cdot\phi(\mathbf{x}_j)$. The kernel trick tells us:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)\cdot\phi(\mathbf{x}_j) = \exp(-\gamma\|\mathbf{x}_i - \mathbf{x}_j\|^2)$$

The right-hand side is a simple scalar computation in the **original** $d$-dimensional space. We never construct $\phi(\mathbf{x})$.

**Computational cost:**
- Computing the full Gram matrix: $O(n^2)$ kernel evaluations, each $O(d)$ → total $O(n^2 d)$
- Solving the QP: $O(n^3)$ in the worst case
- Prediction: $O(n_{sv} \cdot d)$ per new email

None of these costs involve the dimensionality of the feature space — they all scale with $n$ and $d$ (original dimensions).

**Why the Taylor expansion helps:** $\exp(-\gamma\|\mathbf{x}-\mathbf{z}\|^2) = \exp(-\gamma\|\mathbf{x}\|^2)\exp(-\gamma\|\mathbf{z}\|^2)\exp(2\gamma\,\mathbf{x}\cdot\mathbf{z})$, and expanding $\exp(2\gamma\,\mathbf{x}\cdot\mathbf{z})$ as a Taylor series gives the infinite-dimensional dot product — but we never compute it explicitly.

</details>

---

### Q2
$K(\mathbf{x}, \mathbf{z}) = (\mathbf{x}\cdot\mathbf{z} + 1)^2$ is a valid kernel (PSD matrix). Is $K(\mathbf{x}, \mathbf{z}) = \exp(\mathbf{x}\cdot\mathbf{z})$ a valid kernel? How would you check?

<details>
<summary>Show Answer</summary>

**Yes, $K(\mathbf{x}, \mathbf{z}) = \exp(\mathbf{x}\cdot\mathbf{z})$ is a valid kernel.**

**How to check (three approaches):**

**1. Taylor expansion:**
$$\exp(\mathbf{x}\cdot\mathbf{z}) = \sum_{k=0}^{\infty} \frac{(\mathbf{x}\cdot\mathbf{z})^k}{k!}$$
Each term $(\mathbf{x}\cdot\mathbf{z})^k$ corresponds to the polynomial kernel of degree $k$ (with appropriate scaling) — which is a valid kernel. A non-negative weighted sum of valid kernels is also a valid kernel. So $\exp(\mathbf{x}\cdot\mathbf{z})$ is valid.

**2. Feature space:** This corresponds to the inner product in the space of all infinite-degree polynomial features (with factorial weights). The feature map exists, so it's valid.

**3. Empirical check:** Compute the Gram matrix $G_{ij} = \exp(\mathbf{x}_i\cdot\mathbf{x}_j)$ for any dataset and check that all eigenvalues are ≥ 0. (We did this in Section 7 above — confirmed PSD.)

**Closure properties that make checking easier:**
- $k_1 + k_2$ is valid if $k_1, k_2$ are valid
- $c \cdot k$ is valid if $k$ is valid and $c \geq 0$
- $k_1 \cdot k_2$ is valid if $k_1, k_2$ are valid
- $f(k)$ is valid if $k$ is valid and $f$ has non-negative Taylor coefficients (e.g., $\exp(\cdot)$)

</details>

---

### Q3
You use an RBF kernel with **very large γ**. Each training point is "similar" only to its very close neighbors. What happens to the decision boundary? What type of problem does this cause?

<details>
<summary>Show Answer</summary>

**With very large γ, the decision boundary becomes extremely jagged and overfit to individual training points.**

**Mechanically:** Large γ means $\exp(-\gamma\|\mathbf{x}-\mathbf{z}\|^2) \approx 0$ whenever $\mathbf{x}$ and $\mathbf{z}$ are more than a tiny distance apart. Each training point becomes its own "island" of similarity.

**Effect on the decision function:**
$$f(\mathbf{x}) = \sum_{\text{SVs}} \alpha_i y_i K(\mathbf{x}_i, \mathbf{x}) + b$$
When $\gamma \to \infty$: $K(\mathbf{x}_i, \mathbf{x}) \approx 1$ if $\mathbf{x} = \mathbf{x}_i$, else $\approx 0$. The model essentially memorizes every training point — it becomes a nearest-neighbor classifier.

**What you observe (as seen in Section 6):**
- Training accuracy: 100% (every point classified correctly)
- Decision boundary: scattered islands around each training point
- Number of SVs: approaches n (almost everything becomes an SV)

**The problem: Overfitting.**
- High training accuracy, but terrible test accuracy on unseen emails
- The model has not learned the underlying spam pattern — it has memorized 300 specific emails
- A slightly different spam email (same pattern, different wording) may be misclassified

**Fix:** Use cross-validation to choose γ. The bias-variance tradeoff applies: small γ = high bias (smooth, underfit); large γ = high variance (overfit). Good γ ≈ 1/(2 * median squared pairwise distance) as a starting heuristic.

</details>

## Summary & Key Takeaways

| Concept | Summary |
|---|---|
| **Why kernels?** | Linear SVM fails on non-linearly separable data |
| **Explicit mapping** | φ: $\mathbb{R}^d → \mathbb{R}^D$ creates linear separability, but $D$ can be huge |
| **The trick** | $K(\mathbf{x},\mathbf{z}) = \phi(\mathbf{x})\cdot\phi(\mathbf{z})$ — compute dot product in high-D without going there |
| **Mercer's theorem** | $K$ is valid iff the Gram matrix is positive semi-definite |
| **Polynomial kernel** | $(\mathbf{x}\cdot\mathbf{z}+c)^d$ — degree-$d$ polynomial feature space |
| **RBF kernel** | $\exp(-\gamma\|\mathbf{x}-\mathbf{z}\|^2)$ — infinite-dimensional, locality controlled by $\gamma$ |
| **γ effect** | Small γ → smooth (underfit); Large γ → spiky islands (overfit) |
| **Computational cost** | $O(n^2 d)$ for Gram matrix — independent of feature space dimension |

---

### The Kernel Trick in One Sentence

> **The kernel trick lets you compute dot products in a high (or infinite) dimensional feature space using only computations in the original low-dimensional space — enabling non-linear classification at the cost of evaluating a simple similarity function.**

---

**Congratulations on completing Week 10!** You've covered:
- Notebook 01: Linear SVM — the big-margin classifier
- Notebook 02: Dual formulation and the QP problem
- Notebook 03: Soft-margin SVM and the role of C
- Notebook 04: Hinge loss and regularization
- Notebook 05: Support vectors and their role
- **Notebook 06: The kernel trick — your most powerful tool**